In [1]:
# 🤖 Importación de librerías principales
import pandas as pd
import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import seaborn as sns

import sklearn
import sklearn.linear_model
import sklearn.ensemble
import sklearn.tree
import sklearn.svm
import sklearn.neighbors
import sklearn.preprocessing
import sklearn.pipeline
import sklearn.model_selection
import sklearn.metrics
import sklearn.decomposition
import sklearn.feature_selection
import sklearn.impute
import sklearn.inspection

import joblib

import streamlit as st

import holidays

In [2]:
inferencia_df = pd.read_csv('../data/raw/inferencia/ventas_2025_inferencia.csv')
inferencia_df['fecha'] = pd.to_datetime(inferencia_df['fecha'])

print(f"Shape: {inferencia_df.shape}")
inferencia_df.head()

Shape: (888, 13)


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,Amazon,Decathlon,Deporvillage
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,89.51,113.43,104.78
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,128.73,112.91,122.88
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,84.28,74.51,85.57
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,75.54,70.32,71.13
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,33.84,31.32,34.41


In [3]:

# ============================================================
# VARIABLES TEMPORALES Y CALENDARIO
# ============================================================

años_df     = inferencia_df["fecha"].dt.year.unique().tolist()
festivos_es = holidays.Spain(years=años_df)

def black_friday(year):
    nov1         = pd.Timestamp(year=year, month=11, day=1)
    first_friday = nov1 + pd.Timedelta(days=(4 - nov1.dayofweek) % 7)
    return first_friday + pd.Timedelta(weeks=3)

def cyber_monday(year):
    return black_friday(year) + pd.Timedelta(days=3)

def primer_domingo_mayo(year):
    may1 = pd.Timestamp(year=year, month=5, day=1)
    return may1 + pd.Timedelta(days=(6 - may1.dayofweek) % 7)

def pascua(year):
    a = year % 19; b, c = divmod(year, 100); d, e = divmod(b, 4)
    f_ = (b + 8) // 25; g = (b - f_ + 1) // 3
    h  = (19*a + b - d - g + 15) % 30
    i, k = divmod(c, 4); l = (32 + 2*e + 2*i - h - k) % 7
    m    = (a + 11*h + 22*l) // 451
    month, day = divmod(114 + h + l - 7*m, 31)
    return pd.Timestamp(year=year, month=month, day=day + 1)

def _dias_hasta(fn, d):
    candidato = fn(d.year)
    if candidato < d:
        candidato = fn(d.year + 1)
    return (candidato - d).days

_bf  = {y: black_friday(y)                       for y in años_df}
_cm  = {y: cyber_monday(y)                       for y in años_df}
_dm  = {y: primer_domingo_mayo(y)                for y in años_df}
_dp  = {y: pd.Timestamp(year=y, month=3, day=19) for y in años_df}
_pac = {y: pascua(y)                             for y in años_df}

f = inferencia_df["fecha"]

inferencia_df["año"]        = f.dt.year
inferencia_df["mes"]        = f.dt.month
inferencia_df["trimestre"]  = f.dt.quarter
inferencia_df["semana_año"] = f.dt.isocalendar().week.astype(int)
inferencia_df["dia_mes"]    = f.dt.day
inferencia_df["dia_semana"] = f.dt.dayofweek
inferencia_df["dia_año"]    = f.dt.dayofyear

_nom_dia = {0:"Lunes",1:"Martes",2:"Miércoles",3:"Jueves",
            4:"Viernes",5:"Sábado",6:"Domingo"}
_nom_mes = {1:"Enero",2:"Febrero",3:"Marzo",4:"Abril",5:"Mayo",6:"Junio",
            7:"Julio",8:"Agosto",9:"Septiembre",10:"Octubre",11:"Noviembre",12:"Diciembre"}
inferencia_df["nombre_dia"] = f.dt.dayofweek.map(_nom_dia)
inferencia_df["nombre_mes"] = f.dt.month.map(_nom_mes)

_temp = {12:"Invierno",1:"Invierno",2:"Invierno",
          3:"Primavera",4:"Primavera",5:"Primavera",
          6:"Verano",  7:"Verano",  8:"Verano",
          9:"Otoño",  10:"Otoño",  11:"Otoño"}
inferencia_df["temporada"] = f.dt.month.map(_temp)

inferencia_df["es_fin_semana"] = (f.dt.dayofweek >= 5).astype(int)
inferencia_df["es_lunes"]      = (f.dt.dayofweek == 0).astype(int)
inferencia_df["es_festivo"]    = f.map(lambda d: int(d in festivos_es))

inferencia_df["es_semana_santa"] = f.map(
    lambda d: int(
        d.year in _pac and
        _pac[d.year] - pd.Timedelta(days=7) <= d <= _pac[d.year]
    )
)

inferencia_df["es_black_friday"] = f.map(lambda d: int(d == _bf.get(d.year)))
inferencia_df["es_cyber_monday"] = f.map(lambda d: int(d == _cm.get(d.year)))
inferencia_df["es_dia_madre"]    = f.map(lambda d: int(d == _dm.get(d.year)))
inferencia_df["es_dia_padre"]    = f.map(lambda d: int(d == _dp.get(d.year)))
inferencia_df["es_navidad"]      = ((f.dt.month == 12) & (f.dt.day == 25)).astype(int)
inferencia_df["es_nochebuena"]   = ((f.dt.month == 12) & (f.dt.day == 24)).astype(int)
inferencia_df["es_nochevieja"]   = ((f.dt.month == 12) & (f.dt.day == 31)).astype(int)
inferencia_df["es_reyes"]        = ((f.dt.month ==  1) & (f.dt.day ==  6)).astype(int)
inferencia_df["es_san_valentin"] = ((f.dt.month ==  2) & (f.dt.day == 14)).astype(int)
inferencia_df["es_halloween"]    = ((f.dt.month == 10) & (f.dt.day == 31)).astype(int)

inferencia_df["pre_black_friday"] = f.map(
    lambda d: int(1 <= (_bf.get(d.year) - d).days <= 7)
)
inferencia_df["pre_navidad"] = ((f.dt.month == 12) & f.dt.day.between(18, 24)).astype(int)
inferencia_df["pre_reyes"]   = ((f.dt.month ==  1) & f.dt.day.between(1,  5)).astype(int)

inferencia_df["es_inicio_mes"] = (f.dt.day <= 5).astype(int)
inferencia_df["es_fin_mes"]    = (f.dt.day >= f.dt.days_in_month - 4).astype(int)

inferencia_df["dias_para_black_friday"] = f.map(lambda d: _dias_hasta(black_friday, d))
inferencia_df["dias_para_navidad"]      = f.map(
    lambda d: _dias_hasta(lambda y: pd.Timestamp(year=y, month=12, day=25), d)
)
inferencia_df["dias_para_reyes"] = f.map(
    lambda d: _dias_hasta(lambda y: pd.Timestamp(year=y, month=1, day=6), d)
)

inferencia_df["mes_sin"]        = np.sin(2 * np.pi * f.dt.month     / 12)
inferencia_df["mes_cos"]        = np.cos(2 * np.pi * f.dt.month     / 12)
inferencia_df["dia_semana_sin"] = np.sin(2 * np.pi * f.dt.dayofweek / 7)
inferencia_df["dia_semana_cos"] = np.cos(2 * np.pi * f.dt.dayofweek / 7)
inferencia_df["dia_mes_sin"]    = np.sin(2 * np.pi * f.dt.day       / 31)
inferencia_df["dia_mes_cos"]    = np.cos(2 * np.pi * f.dt.day       / 31)

print(f"✓ Variables temporales: OK — {inferencia_df.shape[1]} columnas")

# ============================================================
# LAGS Y MEDIA MÓVIL
#
# unidades_vendidas es NaN en noviembre (es el target a predecir).
# Los lags de noviembre serán NaN para lag_1 en nov-02, lag_1 y
# lag_2 en nov-03, etc. Solo nov-01 tiene los 7 lags completos
# (procedentes de oct-25 a oct-31, que sí tienen valor real).
#
# NO se usa dropna: HistGradientBoostingRegressor maneja NaN de
# forma nativa; el filtro final por mes==11 eliminará octubre.
# ============================================================

inferencia_df = inferencia_df.sort_values(["producto_id", "año", "fecha"]).reset_index(drop=True)

for i in range(1, 8):
    inferencia_df[f"lag_{i}"] = (
        inferencia_df.groupby(["producto_id", "año"])["unidades_vendidas"]
        .shift(i)
    )

inferencia_df["media_movil_7"] = (
    inferencia_df.groupby(["producto_id", "año"])["unidades_vendidas"]
    .transform(lambda x: x.shift(1).rolling(7, min_periods=7).mean())
)

lag_cols    = [f"lag_{i}" for i in range(1, 8)]
nuevas_cols = lag_cols + ["media_movil_7"]

nov_mask = inferencia_df["mes"] == 11
print(f"✓ Lags y media móvil: OK — {inferencia_df.shape[0]:,} filas totales (oct + nov)")
print(f"\n  NaN en columnas de lag para filas de NOVIEMBRE (esperado por diseño):")
print(inferencia_df.loc[nov_mask, nuevas_cols].isnull().sum().to_string())
print(f"\n  → nov-01 tiene los 7 lags completos desde octubre.")
print(f"  → nov-02 en adelante tienen lags parciales / NaN (HGBR los maneja internamente).")

# ============================================================
# VARIABLE DE DESCUENTO
# ============================================================

inferencia_df["descuento_porcentaje"] = (
    (inferencia_df["precio_venta"] - inferencia_df["precio_base"]) / inferencia_df["precio_base"] * 100
)
print(f"\n✓ descuento_porcentaje: OK")

# ============================================================
# VARIABLES DE COMPETENCIA
# ============================================================

inferencia_df["precio_competencia"] = inferencia_df[["Amazon", "Decathlon", "Deporvillage"]].mean(axis=1)
inferencia_df["ratio_precio"]       = inferencia_df["precio_venta"] / inferencia_df["precio_competencia"]
inferencia_df.drop(columns=["Amazon", "Decathlon", "Deporvillage"], inplace=True)

print(f"✓ precio_competencia + ratio_precio: OK  |  Eliminadas: Amazon, Decathlon, Deporvillage")

# ============================================================
# ONE-HOT ENCODING
# ============================================================

cols_cat = ["nombre", "categoria", "subcategoria"]
for col in cols_cat:
    inferencia_df[f"{col}_h"] = inferencia_df[col]

ohe_cols      = [f"{col}_h" for col in cols_cat]
inferencia_df = pd.get_dummies(inferencia_df, columns=ohe_cols, drop_first=False, dtype=int)

nuevas_ohe = [c for c in inferencia_df.columns if any(c.startswith(f"{col}_h_") for col in cols_cat)]
print(f"✓ OHE: {len(nuevas_ohe)} columnas generadas")
print(f"\nShape tras todas las transformaciones: {inferencia_df.shape}")

# ============================================================
# FILTRAR SOLO NOVIEMBRE Y GUARDAR
# ============================================================

inferencia_df = inferencia_df[inferencia_df["mes"] == 11].reset_index(drop=True)

output_path = "../data/processed/inferencia_df_transformado.csv"
inferencia_df.to_csv(output_path, index=False)

print(f"\n✓ Solo noviembre 2025: {len(inferencia_df):,} registros ({len(inferencia_df)//30} productos × 30 días)")
print(f"✓ Guardado en        : {output_path}")
print(f"  Shape final        : {inferencia_df.shape}")
inferencia_df.head()


✓ Variables temporales: OK — 51 columnas
✓ Lags y media móvil: OK — 888 filas totales (oct + nov)

  NaN en columnas de lag para filas de NOVIEMBRE (esperado por diseño):
lag_1            696
lag_2            672
lag_3            648
lag_4            624
lag_5            600
lag_6            576
lag_7            552
media_movil_7    696

  → nov-01 tiene los 7 lags completos desde octubre.
  → nov-02 en adelante tienen lags parciales / NaN (HGBR los maneja internamente).

✓ descuento_porcentaje: OK
✓ precio_competencia + ratio_precio: OK  |  Eliminadas: Amazon, Decathlon, Deporvillage
✓ OHE: 44 columnas generadas

Shape tras todas las transformaciones: (888, 103)

✓ Solo noviembre 2025: 720 registros (24 productos × 30 días)
✓ Guardado en        : ../data/processed/inferencia_df_transformado.csv
  Shape final        : (720, 103)


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,...,subcategoria_h_Esterilla Yoga,subcategoria_h_Mancuernas Ajustables,subcategoria_h_Mochila Trekking,subcategoria_h_Pesa Rusa,subcategoria_h_Pesas Casa,subcategoria_h_Rodillera Yoga,subcategoria_h_Ropa Montaña,subcategoria_h_Ropa Running,subcategoria_h_Zapatillas Running,subcategoria_h_Zapatillas Trail
0,2025-11-01,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,...,0,0,0,0,0,0,0,0,1,0
1,2025-11-02,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,...,0,0,0,0,0,0,0,0,1,0
2,2025-11-03,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,...,0,0,0,0,0,0,0,0,1,0
3,2025-11-04,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,...,0,0,0,0,0,0,0,0,1,0
4,2025-11-05,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,...,0,0,0,0,0,0,0,0,1,0
